# Link Prediction — Exercises

Companion to the note [Link Prediction](Link%20Prediction.md).

Practise link prediction end to end: neighbourhood heuristics (common neighbours, Jaccard, Adamic–Adar, preferential attachment), the path-based Katz index, shallow embeddings with a dot-product decoder, a learned combiner, and a leak-free evaluation protocol with ROC-AUC, AP and Hits@k. Graphs are NumPy adjacency matrices.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)

def from_edges(n, edges):
    A = np.zeros((n, n))
    for i, j in edges:
        A[i, j] = A[j, i] = 1
    return A

# Small graph for the by-hand exercises
E_lp = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 4), (2, 4), (3, 5), (4, 5), (4, 6)]
A_lp = from_edges(7, E_lp)

# A random geometric graph: 250 points in the unit square, connected when closer than 0.1.
# Such graphs have strong triadic closure, like many real social / spatial networks.
pts = rng.random((250, 2))
_dist = np.linalg.norm(pts[:, None] - pts[None], axis=-1)
A_geo = ((_dist < 0.1) & (_dist > 0)).astype(float)

## Part A · Concepts

### Exercise 1 · Why do heuristics work?
*🧠 Concept · ★☆☆*

(a) Which sociological principle makes **common neighbours** a good predictor of future friendships?
(b) Why does **Adamic–Adar** down-weight common neighbours with high degree?
(c) **Preferential attachment** $d_ud_v$ ignores the neighbourhoods entirely. On which kind of network can it still work well, and when is it useless?
(d) Why are **Katz/PageRank**-type scores called *global*?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of triadic closure, of a celebrity who is everybody's 'common friend', and of the 'rich get richer' growth of networks.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Triadic closure**: two people with a mutual friend are likely to meet and become friends (plus homophily).
(b) A shared neighbour with huge degree (a celebrity, a hub airport) says little about $u$ and $v$; a shared neighbour with degree 2 is very informative. $1/\log d_w$ weights rare connections more (similar in spirit to IDF).
(c) In growing scale-free networks where hubs attract new links (citation networks, the web, protein interactions). Useless when degree is not predictive, e.g. random geometric or lattice-like graphs.
(d) They count paths of **all** lengths (Katz: $\sum_\ell\beta^\ell (A^\ell)_{uv}$), so they use the whole graph, not just the 1-hop neighbourhoods. They can score pairs without any common neighbour.

</details>

### Exercise 2 · Leakage in link prediction
*🧠 Concept · ★★☆*

The note warns: *remove test edges from the message-passing graph.* Explain concretely what goes wrong if you
(a) compute common-neighbour scores, (b) compute a spectral/MF embedding, or (c) run a GNN encoder on the **full** graph, and then evaluate on held-out edges.
How should the split be organised for training, validation and test? (You will measure the effect in Exercise 16.)

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

If the edge $(u,v)$ is still in $A$, then $u \in N(v)$; and the embedding was fitted to reproduce $A_{uv}=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) With $(u,v)$ present, $u$ and $v$ are each other's neighbours, which inflates the scores of their neighbourhood-based features (and for degree-based scores, $d_u$ and $d_v$ include the test edge).
(b) The embedding is fitted to the matrix that *contains* $A_{uv}=1$, so $z_u^\top z_v$ is large partly because it was trained to be: we evaluate on training data.
(c) A GNN aggregates over the test edge itself, so it can trivially "see" the answer.

Protocol: split edges into train / val / test **before** computing anything; the message-passing (input) graph contains only train edges; supervision edges
for training are a subset of train edges (or a disjoint split of them); validation and test positives and their sampled negatives are never in the input graph.
For temporal graphs, split by time (train on the past, predict future edges).

</details>

### Exercise 3 · Negatives and metrics
*🧠 Concept · ★★☆*

A graph with $n = 10^5$ nodes has $10^6$ edges. (a) Roughly what fraction of all node pairs are edges? (b) Why do we **sample** negative non-edges
for evaluation, and why does the choice of negatives (uniform random vs "hard" negatives at distance 2) matter? (c) When is ROC-AUC misleading and
AP or Hits@k more informative?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

There are $\binom{n}{2}\approx 5\cdot10^9$ pairs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $10^6 / 5\cdot 10^9 = 2\cdot10^{-4}$, i.e. 0.02%: extreme class imbalance.
(b) Scoring all $5\cdot10^9$ pairs is infeasible, so we sample. Uniform negatives are mostly far apart and trivially easy (any heuristic separates them), which inflates AUC;
hard negatives (e.g. 2-hop pairs) test whether the model can rank *plausible* candidates, closer to the real recommendation task.
(c) ROC-AUC treats all positive–negative pairs equally and stays high even if the top of the ranking is poor; with heavy imbalance, AP (precision-focused) and Hits@k
(is the true edge ranked above the $k$-th best negative?) reflect what users of a ranking see. See [[Model Evaluation and Metrics]].

</details>

## Part B · By hand

### Exercise 4 · Four heuristic scores
*✍️ By hand · ★☆☆*

`A_lp` has 7 nodes and edges $\{0,1\},\{0,2\},\{0,3\},\{1,2\},\{1,4\},\{2,4\},\{3,5\},\{4,5\},\{4,6\}$.
For the non-edge $(0,4)$ compute common neighbours, Jaccard, Adamic–Adar (natural log) and preferential attachment.
Then compute Adamic–Adar for $(3,4)$. Which pair is more likely to link according to each score?

In [ ]:
cn_04 = None
jac_04 = None
aa_04 = None
pa_04 = None
aa_34 = None

def _nb(u): return set(np.flatnonzero(A_lp[u]))
_d = A_lp.sum(1)
check('CN(0,4)', cn_04, len(_nb(0) & _nb(4)))
check('Jaccard(0,4)', jac_04, len(_nb(0) & _nb(4)) / len(_nb(0) | _nb(4)))
check('AA(0,4)', aa_04, sum(1 / np.log(_d[w]) for w in _nb(0) & _nb(4)), tol=1e-3)
check('PA(0,4)', pa_04, _d[0] * _d[4])
check('AA(3,4)', aa_34, sum(1 / np.log(_d[w]) for w in _nb(3) & _nb(4)), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$N(0) = \{1,2,3\}$, $N(4) = \{1,2,5,6\}$, $N(3) = \{0, 5\}$. Degrees: $d_1 = d_2 = 3$, $d_5 = 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(0,4)$: $N(0)\cap N(4) = \{1,2\}$ → CN $=2$; $N(0)\cup N(4) = \{1,2,3,5,6\}$ → Jaccard $= 2/5 = 0.4$;
AA $= \frac{1}{\ln 3}+\frac{1}{\ln 3} = 2/\ln 3\approx 1.820$; PA $= d_0d_4 = 3\cdot4 = 12$.

$(3,4)$: the only common neighbour is node 5 with $d_5 = 2$: AA $= 1/\ln 2\approx 1.443$ (CN = 1, Jaccard = 1/5, PA = 8).
All four scores prefer $(0,4)$, but AA narrows the gap (ratio 1.26 instead of CN's 2) because node 5 is a low-degree, "informative" common neighbour.

```python
cn_04 = 2
jac_04 = 0.4
aa_04 = 2 / np.log(3)
pa_04 = 12
aa_34 = 1 / np.log(2)
```

</details>

### Exercise 5 · Walk counts and the Katz index
*✍️ By hand · ★★☆*

The Katz index is $S_{uv} = \sum_{\ell\ge1}\beta^\ell (A^\ell)_{uv}$, i.e. $S = (I-\beta A)^{-1} - I$.
(a) For `A_lp`, count by hand the walks of length 2 and length 3 from 0 to 4, and give the Katz score truncated after $\ell = 3$ for $\beta = 0.1$.
(b) For which $\beta$ does the series converge? (`A_lp` has $\lambda_{\max}\approx 2.866$.)

In [ ]:
walks2_04 = None
walks3_04 = None
katz3_04 = None
beta_max = None

_A2 = A_lp @ A_lp; _A3 = _A2 @ A_lp
check('walks of length 2', walks2_04, _A2[0, 4])
check('walks of length 3', walks3_04, _A3[0, 4])
check('truncated Katz', katz3_04, 0.1**2 * _A2[0, 4] + 0.1**3 * _A3[0, 4])
check('beta_max', beta_max, 1 / np.linalg.eigvalsh(A_lp).max(), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Length 2: through a common neighbour. Length 3: $0\to a\to b\to 4$ with $a\in N(0)$, $b\in N(4)$ and $\{a,b\}\in E$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\sum_\ell (\beta A)^\ell$ converges iff the spectral radius of $\beta A$ is below 1.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Length 2: $0\!-\!1\!-\!4$, $0\!-\!2\!-\!4$ → 2. Length 3: $0\!-\!1\!-\!2\!-\!4$, $0\!-\!2\!-\!1\!-\!4$, $0\!-\!3\!-\!5\!-\!4$ → 3.
Katz (truncated): $0.01\cdot2 + 0.001\cdot3 = 0.023$ (the full series gives $\approx 0.0251$; longer walks add the rest).
(b) Converges iff $\beta < 1/\lambda_{\max}(A) \approx 1/2.866 \approx 0.349$. Small $\beta$ → close to common neighbours; larger $\beta$ → more global.

```python
walks2_04 = 2
walks3_04 = 3
katz3_04 = 0.023
beta_max = 1 / 2.8662
```

</details>

### Exercise 6 · ROC-AUC, AP and Hits@k by hand
*✍️ By hand · ★★☆*

A model scores three held-out positive edges $(0.9, 0.7, 0.4)$ and four sampled negatives $(0.8, 0.3, 0.2, 0.1)$.
Compute (a) ROC-AUC as the fraction of (positive, negative) pairs ranked correctly, (b) average precision
$\text{AP} = \frac1{|P|}\sum_{k:\,\text{pos at rank }k}\text{Precision@}k$, (c) Hits@1 (fraction of positives scored above the **highest** negative, as in OGB).

In [ ]:
auc_hand = None
ap_hand = None
hits1_hand = None

_y = np.r_[1, 1, 1, 0, 0, 0, 0]; _s = np.r_[.9, .7, .4, .8, .3, .2, .1]
check('ROC-AUC', auc_hand, roc_auc_score(_y, _s), tol=1e-3)
check('AP', ap_hand, average_precision_score(_y, _s), tol=1e-3)
check('Hits@1', hits1_hand, np.mean(_s[:3] > np.sort(_s[3:])[-1]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Sort all scores: 0.9 (P), 0.8 (N), 0.7 (P), 0.4 (P), 0.3, 0.2, 0.1 (N). There are $3\cdot4 = 12$ pairs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) 0.9 beats all 4 negatives, 0.7 beats 3, 0.4 beats 3: $10/12 \approx 0.833$.
(b) Positives sit at ranks 1, 3, 4: precisions $1/1, 2/3, 3/4$ → $\text{AP} = (1 + 0.667 + 0.75)/3 \approx 0.806$.
(c) Only 0.9 beats the best negative 0.8: Hits@1 $= 1/3$. (Hits@2 would compare against the 2nd-best negative 0.3 and give $3/3$.)

```python
auc_hand = 10 / 12
ap_hand = (1 + 2/3 + 3/4) / 3
hits1_hand = 1 / 3
```

</details>

### Exercise 7 · Dot-product decoder and its gradient
*✍️ By hand · ★★☆*

An embedding model predicts $p(u\sim v) = \sigma(z_u^\top z_v)$ and is trained with binary cross-entropy.
For the **positive** edge with $z_u = (1, 2)$, $z_v = (0.5, -1)$: compute the predicted probability, the loss $-\log\sigma(z_u^\top z_v)$
and the gradient of the loss w.r.t. $z_u$. In which direction does a gradient step move $z_u$?

In [ ]:
p_edge = None
loss_edge = None
grad_zu = None

_zu, _zv = np.array([1., 2.]), np.array([.5, -1.])
_f = lambda zu: -np.log(1 / (1 + np.exp(-zu @ _zv)))
check('probability', p_edge, 1 / (1 + np.exp(-_zu @ _zv)), tol=1e-4)
check('loss', loss_edge, _f(_zu), tol=1e-4)
check('gradient', grad_zu, [(_f(_zu + e) - _f(_zu - e)) / 2e-6 for e in 1e-6 * np.eye(2)], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\frac{d}{dx}[-\log\sigma(x)] = \sigma(x) - 1$; chain rule with $x = z_u^\top z_v$, $\partial x/\partial z_u = z_v$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$z_u^\top z_v = 0.5 - 2 = -1.5$, $p = \sigma(-1.5)\approx 0.182$, loss $= -\ln 0.182 \approx 1.701$.
$\nabla_{z_u} = (\sigma(-1.5) - 1)\,z_v \approx -0.818\cdot(0.5,-1) = (-0.409,\ 0.818)$.
Gradient **descent** moves $z_u$ by $-\eta\nabla = +\eta\,0.818\,z_v$, i.e. towards $z_v$, increasing their dot product. For a negative pair the gradient is $\sigma(x)z_v$ and pushes $z_u$ away.

```python
p_edge = 1 / (1 + np.exp(1.5))
loss_edge = np.log(1 + np.exp(1.5))
grad_zu = (p_edge - 1) * np.array([0.5, -1.0])
```

</details>

### Exercise 8 · Where preferential attachment comes from
*✍️ By hand · ★★★*

In the **configuration model** a graph with degree sequence $d_1,\dots,d_n$ (and $2m = \sum_i d_i$ edge stubs) is formed by pairing stubs uniformly at random.
Show that the expected number of edges between $u\neq v$ is approximately $\frac{d_ud_v}{2m}$. For a graph with $m = 1000$ edges, what is this for
$d_u = 40$, $d_v = 25$? How does this justify PA as a score, and why is it the "null model" that modularity compares against?

In [ ]:
expected_edges = None

check('expected edges', expected_edges, 40 * 25 / 2000)

<details>
<summary><b>💡 Hint</b></summary>

Each of $u$'s $d_u$ stubs is paired with one of the other $2m-1$ stubs uniformly; $d_v$ of them belong to $v$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each of $u$'s $d_u$ stubs connects to one of $v$'s $d_v$ stubs with probability $\frac{d_v}{2m-1}$, so $\mathbb E[\#\text{edges}(u,v)] = \frac{d_ud_v}{2m-1}\approx\frac{d_ud_v}{2m}$.
With $m=1000$: $40\cdot25/2000 = 0.5$.

So if links formed with no structure other than degrees, the link probability would be proportional to $d_ud_v$: PA is the optimal score under this null model.
Modularity $Q = \frac1{2m}\sum_{ij}\big(A_{ij} - \frac{d_id_j}{2m}\big)\mathbb 1[c_i = c_j]$ counts how many *more* edges lie within communities than this model predicts.

```python
expected_edges = 0.5
```

</details>

## Part C · Code: heuristics and an honest evaluation

### Exercise 9 · Vectorised heuristic score matrices
*💻 Code · ★★☆*

Implement `heuristic_scores(A)` returning a dict of $n\times n$ score matrices `'cn'`, `'jaccard'`, `'aa'`, `'pa'` computed with matrix operations
(no Python loops over pairs). For Adamic–Adar, use weight $1/\log d_w$ for $d_w\ge2$ (a common neighbour always has $d_w\ge2$).

In [ ]:
def heuristic_scores(A):
    # TODO
    return None

S_lp = heuristic_scores(A_lp)

def _brute(A, u, v):
    Nu, Nv = set(np.flatnonzero(A[u])), set(np.flatnonzero(A[v])); d = A.sum(1)
    return [len(Nu & Nv), len(Nu & Nv) / max(len(Nu | Nv), 1), sum(1 / np.log(d[w]) for w in Nu & Nv), d[u] * d[v]]
_pairs = [(u, v) for u in range(7) for v in range(7) if u != v]
_ref = np.array([_brute(A_lp, u, v) for u, v in _pairs])
_got = None if S_lp is None else np.array([[S_lp[k][u, v] for k in ('cn', 'jaccard', 'aa', 'pa')] for u, v in _pairs])
check('all heuristic scores', _got, _ref)

<details>
<summary><b>💡 Hint 1</b></summary>

CN $= A^2$; PA $= dd^\top$; AA $= A\,\mathrm{diag}(w)\,A$ with $w_i = 1/\log d_i$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$|N(u)\cup N(v)| = d_u + d_v - |N(u)\cap N(v)|$; guard against $0/0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All four are 2-hop computations: $A^2$ costs $O(n^3)$ densely but only $O(\sum_w d_w^2)$ with sparse matrices. In practice one computes scores only
for the candidate pairs being evaluated (rows of $A$ restricted to $u$ and $v$).

```python
def heuristic_scores(A):
    d = A.sum(1)
    cn = A @ A
    union = d[:, None] + d[None, :] - cn
    w = np.where(d >= 2, 1 / np.log(np.maximum(d, 2)), 0.0)
    return {
        'cn': cn,
        'jaccard': np.divide(cn, union, out=np.zeros_like(cn), where=union > 0),
        'aa': (A * w) @ A,
        'pa': np.outer(d, d),
    }

S_lp = heuristic_scores(A_lp)
```

</details>

### Exercise 10 · A leak-free edge split
*💻 Code · ★★☆*

Implement `split_edges(A, test_frac=0.1, seed=0)` that
1. hides a random `test_frac` of the edges (returned as an array `pos_test` of shape `(k, 2)` with $u<v$),
2. samples the same number of **non-edges** of the full graph as `neg_test` (no self-loops, no duplicates),
3. returns `A_train` = the adjacency with the test edges removed.

Apply it to the geometric graph `A_geo`.

In [ ]:
def split_edges(A, test_frac=0.1, seed=0):
    # TODO
    return None, None, None

A_train, pos_test, neg_test = split_edges(A_geo)

if A_train is not None:
    check('test edges removed from the training graph', A_train[pos_test[:, 0], pos_test[:, 1]].sum(), 0)
    check('test positives are real edges', A_geo[pos_test[:, 0], pos_test[:, 1]].min(), 1)
    check('negatives are non-edges, no self-loops', (A_geo[neg_test[:, 0], neg_test[:, 1]].max(), (neg_test[:, 0] == neg_test[:, 1]).any()), (0, False))
    check('balanced and symmetric', (len(pos_test) == len(neg_test), np.allclose(A_train, A_train.T), A_geo.sum() - A_train.sum() == 2 * len(pos_test)), (True, True, True))
else:
    check('edge split', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`iu, ju = np.triu_indices(n, 1)`; positives are the pairs with `A[iu, ju] == 1`, candidates for negatives those with `0`. Use `rng.choice(..., replace=False)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sampling negatives from the upper triangle of $A$'s zero entries guarantees validity. One subtlety: some removed test edges may isolate a node in `A_train`;
heuristics then score it 0, which is the honest outcome (a "cold-start" node).

```python
def split_edges(A, test_frac=0.1, seed=0):
    r = np.random.default_rng(seed)
    n = len(A)
    iu, ju = np.triu_indices(n, 1)
    is_edge = A[iu, ju] == 1
    edges = np.c_[iu[is_edge], ju[is_edge]]
    non_edges = np.c_[iu[~is_edge], ju[~is_edge]]
    k = int(round(test_frac * len(edges)))
    pos = edges[r.choice(len(edges), k, replace=False)]
    neg = non_edges[r.choice(len(non_edges), k, replace=False)]
    A_tr = A.copy()
    A_tr[pos[:, 0], pos[:, 1]] = A_tr[pos[:, 1], pos[:, 0]] = 0
    return A_tr, pos, neg

A_train, pos_test, neg_test = split_edges(A_geo)
print(len(pos_test), 'test edges')
```

</details>

### Exercise 11 · ROC-AUC from scratch and a heuristic leaderboard
*💻 Code · ★★☆*

Implement `auc(pos_scores, neg_scores)` as the probability that a random positive outscores a random negative (ties count ½).
Score the test pairs with each heuristic computed on **`A_train`** and store the AUCs in `auc_table = {'cn': ..., 'jaccard': ..., 'aa': ..., 'pa': ...}`.
Which heuristic is best on this geometric graph, and why is PA weak here?

In [ ]:
def auc(pos_scores, neg_scores):
    # TODO
    return None

auc_table = None

_p, _n = rng.normal(1, 1, 50), rng.normal(0, 1, 70); _n[:5] = _p[:5]   # include ties
check('auc vs sklearn', auc(_p, _n), roc_auc_score(np.r_[np.ones(50), np.zeros(70)], np.r_[_p, _n]))
if auc_table is not None:
    _S = A_train @ A_train
    check('CN AUC', auc_table['cn'], roc_auc_score(np.r_[np.ones(len(pos_test)), np.zeros(len(neg_test))],
                                                    np.r_[_S[pos_test[:, 0], pos_test[:, 1]], _S[neg_test[:, 0], neg_test[:, 1]]]))
else:
    check('CN AUC', None, 1)

<details>
<summary><b>💡 Hint</b></summary>

Broadcast: `(pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean()` (Mann–Whitney U statistic).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The neighbourhood-overlap scores (CN, Jaccard, AA) reach AUC ≈ 0.98 on this graph, because two points that share neighbours are close in space,
and close points are linked. PA is much weaker (≈ 0.57, barely above chance): it captures only that dense regions have more links, not *which* pairs are close.
On a scale-free citation graph the ranking would look different, which is why you always compare several baselines.

```python
def auc(pos_scores, neg_scores):
    p, n = np.asarray(pos_scores)[:, None], np.asarray(neg_scores)[None, :]
    return (p > n).mean() + 0.5 * (p == n).mean()

S_train = heuristic_scores(A_train)
auc_table = {k: auc(S[pos_test[:, 0], pos_test[:, 1]], S[neg_test[:, 0], neg_test[:, 1]]) for k, S in S_train.items()}
print({k: round(v, 3) for k, v in auc_table.items()})
```

</details>

### Exercise 12 · Average precision and Hits@k
*💻 Code · ★★☆*

Implement `average_precision(pos_scores, neg_scores)` (sort descending, average the precision at the rank of each positive; assume no ties)
and `hits_at_k(pos_scores, neg_scores, k)` (fraction of positives that score strictly above the $k$-th highest negative).
Evaluate Adamic–Adar on the test split; store `ap_aa` and `hits10_aa`.

In [ ]:
def average_precision(pos_scores, neg_scores):
    return None
def hits_at_k(pos_scores, neg_scores, k):
    return None

ap_aa = None
hits10_aa = None

_p, _n = rng.random(40) + 0.3, rng.random(60)
check('AP vs sklearn', average_precision(_p, _n), average_precision_score(np.r_[np.ones(40), np.zeros(60)], np.r_[_p, _n]))
check('Hits@k on the by-hand example', hits_at_k(np.r_[.9, .7, .4], np.r_[.8, .3, .2, .1], 2), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`order = np.argsort(-scores)`, `y_sorted = labels[order]`, `prec = np.cumsum(y_sorted) / np.arange(1, N+1)`, `AP = prec[y_sorted == 1].mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

AP is the area under the precision–recall curve (step-wise); Hits@k depends only on how the positives compare with the *hardest* few negatives,
so it is much stricter than AUC when there are many negatives (OGB uses up to 100k negatives per positive).

```python
def average_precision(pos_scores, neg_scores):
    s = np.r_[pos_scores, neg_scores]; y = np.r_[np.ones(len(pos_scores)), np.zeros(len(neg_scores))]
    ys = y[np.argsort(-s, kind='stable')]
    prec = np.cumsum(ys) / np.arange(1, len(ys) + 1)
    return prec[ys == 1].mean()

def hits_at_k(pos_scores, neg_scores, k):
    thr = np.sort(neg_scores)[-k]
    return np.mean(np.asarray(pos_scores) > thr)

_aa = S_train['aa']
_ps, _ns = _aa[pos_test[:, 0], pos_test[:, 1]], _aa[neg_test[:, 0], neg_test[:, 1]]
ap_aa, hits10_aa = average_precision(_ps, _ns), hits_at_k(_ps, _ns, 10)
print('AP =', round(ap_aa, 3), ' Hits@10 =', round(hits10_aa, 3))
```

</details>

### Exercise 13 · Katz index
*💻 Code · ★★☆*

Implement `katz(A, beta)` $= (I - \beta A)^{-1} - I$ and evaluate its AUC on the test split with $\beta = 0.5/\lambda_{\max}(A_{\text{train}})$.
Store it in `auc_katz`. Also confirm numerically that the closed form equals the truncated series $\sum_{\ell=1}^{200}\beta^\ell A^\ell$.

In [ ]:
def katz(A, beta):
    return None

auc_katz = None

_b = 0.5 / np.linalg.eigvalsh(A_lp).max()
_ser, _P = np.zeros((7, 7)), np.eye(7)
for _ in range(200):
    _P = _b * _P @ A_lp; _ser += _P
check('Katz closed form = series', katz(A_lp, _b), _ser)
check('Katz AUC > 0.9', None if auc_katz is None else auc_katz > 0.9, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.inv(np.eye(n) - beta * A) - np.eye(n)`; for large graphs solve only for the needed columns.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Katz scores pairs even without common neighbours (via longer walks), so it breaks the many ties that CN has at 0. On this geometric graph its AUC is close to the
neighbourhood heuristics; on sparse graphs with many 0-CN test pairs it usually wins.

```python
def katz(A, beta):
    n = len(A)
    return np.linalg.inv(np.eye(n) - beta * A) - np.eye(n)

beta_k = 0.5 / np.linalg.eigvalsh(A_train).max()
K = katz(A_train, beta_k)
auc_katz = auc(K[pos_test[:, 0], pos_test[:, 1]], K[neg_test[:, 0], neg_test[:, 1]])
print('Katz AUC:', round(auc_katz, 3))
```

</details>

### Exercise 14 · Shallow embeddings by matrix factorisation
*💻 Code · ★★★*

Build node embeddings from **`A_train`** by a rank-$k$ eigendecomposition: $A_{\text{train}}\approx U_k\Lambda_kU_k^\top$ with the $k$ largest eigenvalues.
Score $s_{uv} = z_u^\top \Lambda_k z_v$ with $z_u = $ row $u$ of $U_k$ (an *indefinite* dot-product decoder). Compute the test AUC for
$k\in\{2, 8, 32, 128\}$ and store it in `auc_mf` (dict). Then answer: why does the score of $A_{\text{train}}$'s own low-rank reconstruction tell us anything about **missing** edges?

In [ ]:
auc_mf = None  # {k: auc}

if auc_mf is not None:
    check('some rank gives AUC > 0.85', max(auc_mf.values()) > 0.85, True)
else:
    check('some rank gives AUC > 0.85', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`w, U = np.linalg.eigh(A_train)` sorts eigenvalues ascending; take the last $k$ columns. The full score matrix is `U_k @ np.diag(w_k) @ U_k.T`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A low-rank approximation cannot reproduce every individual entry; it smooths $A$ toward a structured matrix, so a missing edge between two nodes with similar
neighbourhoods gets a high reconstructed value (*matrix completion*). Too small $k$ underfits (blurry); too large $k$ reproduces $A_{\text{train}}$ including its
zeros at the hidden edges, so there is a sweet spot (here in the tens). DeepWalk/node2vec do the same with a PMI-transformed walk co-occurrence matrix ([[Graph ML Overview]]).

```python
w_tr, U_tr = np.linalg.eigh(A_train)
auc_mf = {}
for k in (2, 8, 32, 128):
    Uk, wk = U_tr[:, -k:], w_tr[-k:]
    S_mf = (Uk * wk) @ Uk.T
    auc_mf[k] = auc(S_mf[pos_test[:, 0], pos_test[:, 1]], S_mf[neg_test[:, 0], neg_test[:, 1]])
print({k: round(v, 3) for k, v in auc_mf.items()})
```

</details>

### Exercise 15 · Learning to combine heuristics
*💻 Code · ★★★*

Train a logistic regression on heuristic features `[cn, jaccard, aa, log(1+pa), katz]` to predict links. Do it **without leakage**:
split `A_train` once more into `A_inner` + validation positives (10%) with sampled negatives, compute features on `A_inner`, fit on the validation pairs,
then compute features on `A_train` for the test pairs and report the test AUC as `auc_lr`.

In [ ]:
auc_lr = None

check('learned combiner AUC > 0.9', None if auc_lr is None else auc_lr > 0.9, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Reuse `split_edges(A_train, 0.1, seed=1)`. Write `features(A, pairs)` that stacks the five scores for an array of pairs.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Standardise features (or use `LogisticRegression(max_iter=1000)` with `StandardScaler` in a pipeline); features fitted on `A_inner` are slightly smaller in scale than on `A_train`, which is the usual small distribution shift.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The combiner learns, e.g., that AA and Katz matter most and PA a little. Its gain over the best single heuristic is small on this easy graph (≈ 0.99 vs 0.98), but the protocol
is what matters: the model is fitted on pairs whose features never saw those edges, exactly mirroring the test situation. SEAL and GNN link predictors
([[Graph Neural Networks]]) generalise this idea by *learning* the heuristic from the enclosing subgraph.

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def features(A, pairs):
    S = heuristic_scores(A)
    K = katz(A, 0.5 / np.linalg.eigvalsh(A).max())
    u, v = pairs[:, 0], pairs[:, 1]
    return np.c_[S['cn'][u, v], S['jaccard'][u, v], S['aa'][u, v], np.log1p(S['pa'][u, v]), K[u, v]]

A_inner, pos_val, neg_val = split_edges(A_train, 0.1, seed=1)
X_fit = features(A_inner, np.r_[pos_val, neg_val])
y_fit = np.r_[np.ones(len(pos_val)), np.zeros(len(neg_val))]
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_fit, y_fit)
X_test = features(A_train, np.r_[pos_test, neg_test])
y_test = np.r_[np.ones(len(pos_test)), np.zeros(len(neg_test))]
auc_lr = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print('learned combiner AUC:', round(auc_lr, 3))
```

</details>

### Exercise 16 · Measuring leakage
*💻 Code · ★★☆*

Repeat the **preferential attachment** and **Katz** evaluations, but compute the scores on the **full** graph `A_geo` (test edges *not* removed).
Store the leaky AUCs in `auc_leaky = {'pa': ..., 'katz': ...}` and compare with the honest numbers from Exercises 11 and 13. Explain the difference.

In [ ]:
auc_leaky = None

if auc_leaky is not None:
    _d = A_train.sum(1); _pa = np.outer(_d, _d)
    _honest_pa = roc_auc_score(np.r_[np.ones(len(pos_test)), np.zeros(len(neg_test))], np.r_[_pa[pos_test[:, 0], pos_test[:, 1]], _pa[neg_test[:, 0], neg_test[:, 1]]])
    check('leaky PA AUC is higher than honest', auc_leaky['pa'] > _honest_pa, True)
else:
    check('leaky PA AUC is higher than honest', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Only the matrix the scores are computed from changes; the test pairs stay the same.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With the test edges left in, every positive pair gets **+1 to both degrees** (PA inflates) and a direct length-1 walk (Katz gains a $\beta$ term, the dominant one),
while negatives are unaffected. The AUCs jump (here PA 0.57 → 0.69, Katz 0.98 → 1.00), so a leaky protocol overstates performance; for learned encoders (MF, GNNs) the effect is even larger.

```python
d_full = A_geo.sum(1); PA_full = np.outer(d_full, d_full)
K_full = katz(A_geo, 0.5 / np.linalg.eigvalsh(A_geo).max())
auc_leaky = {name: auc(M[pos_test[:, 0], pos_test[:, 1]], M[neg_test[:, 0], neg_test[:, 1]]) for name, M in (('pa', PA_full), ('katz', K_full))}
print('leaky:', {k: round(v, 3) for k, v in auc_leaky.items()}, ' honest PA:', round(auc_table['pa'], 3), ' honest Katz:', round(auc_katz, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Link Prediction](Link%20Prediction.md).*